# Lab 06 — Decision Trees and k-Nearest Neighbours

**Class 06 · Topic 5.** Guide: `practice/06_trees_knn.pdf`. Lecture: `slides/06_trees_knn.pdf`. Demos: `01_decision_trees.ipynb`
and `02_knn.ipynb`.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work.

**The rule of this lab.** Trees and KNN have no gradient to descend: they are *algorithms*. You write them with NumPy
(the tree) and `jax.numpy` (the distances of KNN), and compare each with its scikit-learn twin on the **same split and the
same scaling**. The notebook prints the difference; explain it.

In [ ]:
import time
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab06_utils import (
    compare,
    fit_standardizer,
    leaves_of,
    load_breast_cancer,
    load_concrete,
    load_digits,
    load_german_credit,
    plot_curves,
    plot_pred_vs_actual,
    print_tree,
    standardize,
    stratified_split,
)
from scipy import stats
from sklearn.datasets import make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
_ = pl.Config.set_tbl_hide_dataframe_shape(True)

# Part A — Data visualization and evaluation methodology

Three data sets: **breast cancer** (30 measurements, benign or malignant), **German credit** (1000 applications,
61 columns after encoding, 30% bad credits) and **concrete** (1030 mixtures, the strength in MPa). The split is
stratified; the standardization statistics (needed for KNN, harmless for trees) come from the **training** part only.

In [ ]:
Xb, yb, names_b = load_breast_cancer()
Xg, yg, names_g = load_german_credit()
Xc, yc, names_c = load_concrete()
Xd, yd, _ = load_digits()
print(f"breast cancer {Xb.shape}, benign {yb.mean():.1%}; german credit {Xg.shape}, bad {yg.mean():.1%}; concrete {Xc.shape}")

rng = np.random.default_rng(0)
trb, teb = stratified_split(yb, 0.25, rng)
trg, teg = stratified_split(yg, 0.25, rng)
trc, tec = np.random.default_rng(1).permutation(len(yc))[: int(0.75 * len(yc))], np.random.default_rng(1).permutation(len(yc))[int(0.75 * len(yc)):]
print(f"baselines (always the majority class): breast cancer {max(yb[teb].mean(), 1 - yb[teb].mean()):.3f}, "
      f"german credit {max(yg[teg].mean(), 1 - yg[teg].mean()):.3f}; concrete: predict the mean, RMSE {np.sqrt(np.mean((yc[trc].mean() - yc[tec]) ** 2)):.2f} MPa")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
j = names_b.index("worst_perimeter")
axes[0].hist([Xb[yb == 1, j], Xb[yb == 0, j]], bins=25, stacked=True, label=["benign", "malignant"])
axes[0].set(xlabel="worst_perimeter", title="breast cancer: one feature, two classes")
axes[0].legend()
j2 = names_g.index("duration")
axes[1].hist([Xg[yg == 0, j2], Xg[yg == 1, j2]], bins=20, stacked=True, label=["good", "bad"])
axes[1].set(xlabel="duration (months)", title="german credit")
axes[1].legend()
axes[2].scatter(Xc[:, names_c.index("age")], yc, s=6)
axes[2].set(xlabel="age (days)", ylabel="strength (MPa)", title="concrete: a non-linear relation")
plt.tight_layout()
plt.show()

# Part B — Impurity and the best split

A node holds the examples that reached it, with class counts $n_1,\dots,n_K$ ($p_k = n_k/n$). Its impurity is the
**Gini index** $G = 1 - \sum_k p_k^2$ or the **entropy** $H = -\sum_k p_k\log_2 p_k$ (with $0\log 0 = 0$). The **gain** of a
question "$x \le t$" is $\ \text{impurity(parent)} - \frac{n_L}{n}\,\text{impurity(left)} - \frac{n_R}{n}\,\text{impurity(right)}$.

## B1. The criteria and the best threshold

Implement `gini(counts)` and `entropy(counts)` for an array whose **last axis** holds the class counts (any number of
leading axes: they are called on a whole matrix of left-child counts at once), and
`best_threshold(x, y, n_classes, criterion)` for one numeric feature `x` and integer labels `y`: sort `x`, take the
cumulative class counts (`np.cumsum` of the one-hot labels), evaluate the gain of **every** split between two *different*
consecutive values at once, and return `(threshold, gain)`, the threshold being the midpoint; `(None, 0.0)` if no split
has a positive gain.

In [ ]:
def gini(counts):
    raise NotImplementedError


def entropy(counts):
    raise NotImplementedError


def best_threshold(x, y, n_classes, criterion=gini):
    raise NotImplementedError

The driver reproduces the table of the lecture (the 14 days of "play tennis?") and compares your best thresholds on the
breast-cancer data with the root split of `DecisionTreeClassifier` (depth 1, same criterion).

In [ ]:
tennis = {"outlook": ["sunny", "sunny", "overcast", "rain", "rain", "rain", "overcast", "sunny", "sunny", "rain", "sunny", "overcast", "overcast", "rain"],
          "humidity": ["high", "high", "high", "high", "normal", "normal", "normal", "high", "normal", "normal", "normal", "high", "normal", "high"],
          "windy": ["no", "yes", "no", "no", "no", "yes", "yes", "no", "no", "no", "yes", "yes", "no", "yes"]}
play = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0])
parent = np.bincount(play, minlength=2)
rows = []
for feat, vals in tennis.items():
    vals = np.array(vals)
    child = sum((vals == v).mean() * entropy(np.bincount(play[vals == v], minlength=2)) for v in np.unique(vals))
    child_g = sum((vals == v).mean() * gini(np.bincount(play[vals == v], minlength=2)) for v in np.unique(vals))
    rows.append((feat, float(entropy(parent) - child), float(gini(parent) - child_g)))
print(f"parent (9 yes, 5 no): entropy {entropy(parent):.3f}, Gini {gini(parent):.3f}")
print(pl.DataFrame(rows, schema=["question", "gain (entropy)", "gain (Gini)"], orient="row"))

rows = []
for name, (X, y, names, tr) in {"breast cancer": (Xb, yb, names_b, trb), "german credit": (Xg, yg, names_g, trg)}.items():
    for crit, fn in (("gini", gini), ("entropy", entropy)):
        gains = [best_threshold(X[tr, j], y[tr], 2, fn) for j in range(X.shape[1])]
        j = int(np.argmax([g for _, g in gains]))
        ref = DecisionTreeClassifier(criterion=crit, max_depth=1, random_state=0).fit(X[tr], y[tr])
        rows.append((name, crit, names[j], gains[j][0], names[ref.tree_.feature[0]], float(ref.tree_.threshold[0])))
print(pl.DataFrame(rows, schema=["data", "criterion", "our feature", "our thr.", "sklearn feature", "sklearn thr."], orient="row"))

# Part C — A tree from scratch

## C1. Grow it and use it

Implement `fit_tree(X, y, n_classes, max_depth, min_leaf, criterion, depth=0)` returning a dictionary node (see
`lab06_utils`): `counts`, `pred` (the majority class), `n`; and, for an internal node, `feature`, `threshold`, `left`,
`right`. Make a **leaf** when the node is pure, the depth limit is reached, or it has fewer than `2 * min_leaf` examples;
otherwise take the best (feature, threshold) over all features with `best_threshold`, **reject** splits that leave fewer
than `min_leaf` examples on a side, and recurse on the two parts. Implement `predict_tree(node, X)`: follow the questions
from the root.

In [ ]:
def fit_tree(X, y, n_classes, max_depth=5, min_leaf=1, criterion=gini, depth=0):
    raise NotImplementedError


def predict_tree(node, X):
    raise NotImplementedError

In [ ]:
rows = []
for name, (X, y, tr, te, names) in {"breast cancer": (Xb, yb, trb, teb, names_b), "german credit": (Xg, yg, trg, teg, names_g)}.items():
    for depth in (2, 4, 6):
        t0 = time.time()
        ours = fit_tree(X[tr], y[tr], 2, max_depth=depth, min_leaf=1)
        t_ours = time.time() - t0
        ref = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X[tr], y[tr])
        pred_ours, pred_ref = predict_tree(ours, X[te]), ref.predict(X[te])
        rows.append((name, depth, float(np.mean(pred_ours == y[te])), float(ref.score(X[te], y[te])), float(np.mean(pred_ours == pred_ref)),
                     leaves_of(ours), int(ref.get_n_leaves()), t_ours))
print(pl.DataFrame(rows, schema=["data", "depth", "acc ours", "acc sklearn", "same pred.", "leaves ours", "leaves sk", "fit (s)"], orient="row"))

tree3 = fit_tree(Xb[trb], yb[trb], 2, max_depth=3)
print("\nOur depth-3 tree for breast cancer, as rules:")
print_tree(tree3, names_b, class_names=["malignant", "benign"])

# Part D — Overfitting and the choice of the depth

## D1. Depth on validation data

Implement `best_depth(Xtr, ytr, Xva, yva, n_classes, depths, min_leaf)`: for each depth fit your tree on the training part
and measure the accuracy on the training and validation parts; return the depth with the best **validation** accuracy (the
smallest one on a tie) and the two curves as arrays.

In [ ]:
def best_depth(Xtr, ytr, Xva, yva, n_classes, depths, min_leaf=1):
    raise NotImplementedError

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
depths = list(range(1, 13))
for ax, (name, X, y, tr, te) in zip(axes, (("breast cancer", Xb, yb, trb, teb), ("german credit", Xg, yg, trg, teg))):
    # the validation part is the second half of the training part; the test part is not touched
    h = len(tr) // 2
    d, tr_acc, va_acc = best_depth(X[tr[:h]], y[tr[:h]], X[tr[h:]], y[tr[h:]], 2, depths)
    ax.plot(depths, tr_acc, label="train")
    ax.plot(depths, va_acc, label="validation")
    ax.axvline(d, color="k", ls=":")
    ax.set(xlabel="max_depth", title=f"{name}: best depth {d}")
    final = fit_tree(X[tr], y[tr], 2, max_depth=d)
    print(f"{name}: best depth {d}; test accuracy of a tree of that depth (fitted on all the training part): {np.mean(predict_tree(final, X[te]) == y[te]):.3f}; "
          f"full tree {np.mean(predict_tree(fit_tree(X[tr], y[tr], 2, max_depth=50), X[te]) == y[te]):.3f}")
axes[0].legend()
plt.tight_layout()
plt.show()

**Cost-complexity pruning** (scikit-learn, given): the whole path of nested trees from one full tree, chosen by 5-fold
cross-validation on the training part.

In [ ]:
rows = []
for name, X, y, tr, te in (("breast cancer", Xb, yb, trb, teb), ("german credit", Xg, yg, trg, teg)):
    full = DecisionTreeClassifier(random_state=0).fit(X[tr], y[tr])
    alphas = np.unique(np.round(full.cost_complexity_pruning_path(X[tr], y[tr]).ccp_alphas, 6))
    grid = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas[:: max(1, len(alphas) // 30)]}, cv=5).fit(X[tr], y[tr])
    rows.append((name, int(full.get_n_leaves()), float(full.score(X[te], y[te])), int(grid.best_estimator_.get_n_leaves()),
                 float(grid.best_params_["ccp_alpha"]), float(grid.score(X[te], y[te]))))
print(pl.DataFrame(rows, schema=["data", "leaves, full", "test acc, full", "leaves, pruned", "best alpha", "test acc, pruned"], orient="row"))

# Part E — A regression tree

## E1. Variance reduction

For a numeric target a leaf predicts the **mean** of its examples and a split is judged by the reduction of the **sum of
squared errors** $\text{SSE} = \sum (y_i - \bar y)^2$. Implement `best_threshold_reg(x, y)` (the same sorted scan, with the
cumulative sums of $y$ and $y^2$: $\text{SSE} = \sum y^2 - (\sum y)^2/n$ for each side) returning `(threshold, gain)`, and
`fit_reg_tree(X, y, max_depth, min_leaf)` returning nodes with `value` (the mean), `pred` (the same), `n`, and
`feature`, `threshold`, `left`, `right` for internal ones; `predict_reg_tree` is given.

In [ ]:
def best_threshold_reg(x, y):
    raise NotImplementedError


def fit_reg_tree(X, y, max_depth=4, min_leaf=5, depth=0):
    raise NotImplementedError


def predict_reg_tree(node, X):
    out = np.empty(len(X))
    for i, x in enumerate(X):
        n = node
        while "feature" in n:
            n = n["left"] if x[n["feature"]] <= n["threshold"] else n["right"]
        out[i] = n["pred"]
    return out

In [ ]:
Xtr_c, Xte_c, ytr_c, yte_c = Xc[trc], Xc[tec], yc[trc], yc[tec]
rows, preds = [], {}
for depth in (2, 4, 6, 10):
    ours = fit_reg_tree(Xtr_c, ytr_c, max_depth=depth, min_leaf=5)
    ref = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=5, random_state=0).fit(Xtr_c, ytr_c)
    p_o, p_r = predict_reg_tree(ours, Xte_c), ref.predict(Xte_c)
    rows.append((depth, float(np.sqrt(np.mean((p_o - yte_c) ** 2))), float(np.sqrt(np.mean((p_r - yte_c) ** 2))), float(np.max(np.abs(p_o - p_r))), leaves_of(ours), int(ref.get_n_leaves())))
    if depth == 6:
        preds = {"ours, depth 6": p_o, "sklearn, depth 6": p_r}
lin = LinearRegression().fit(Xtr_c, ytr_c)
print(pl.DataFrame(rows, schema=["depth", "RMSE ours", "RMSE sklearn", "max |diff|", "leaves ours", "leaves sk"], orient="row"))
print(f"linear regression: RMSE {np.sqrt(np.mean((lin.predict(Xte_c) - yte_c) ** 2)):.2f} MPa")
plot_pred_vs_actual(yte_c, preds)

# Part F — k-Nearest Neighbours with `jax.numpy`

## F1. Distances, neighbours, vote

Implement `pairwise_sq_dists(A, B)` (shape `(len(A), len(B))`, with broadcasting: $\lVert a-b\rVert^2 = \lVert a\rVert^2 +
\lVert b\rVert^2 - 2a^\top b$) and `knn_predict(Xtr, ytr, Xte, k, n_classes, weights="uniform")`: for every test point
take the $k$ nearest training points (`jax.lax.top_k` of the *negative* distances), and let them vote (the sum of the
one-hot labels; with `weights="distance"` each vote is multiplied by $1/(d + 10^{-9})$). Return the winning class.

In [ ]:
def pairwise_sq_dists(A, B):
    raise NotImplementedError


def knn_predict(Xtr, ytr, Xte, k=5, n_classes=2, weights="uniform"):
    raise NotImplementedError

In [ ]:
mu_b, sd_b = fit_standardizer(Xb[trb])
A_b, B_b = standardize(Xb[trb], mu_b, sd_b), standardize(Xb[teb], mu_b, sd_b)
mu_g, sd_g = fit_standardizer(Xg[trg])
A_g, B_g = standardize(Xg[trg], mu_g, sd_g), standardize(Xg[teg], mu_g, sd_g)
D_ours = np.asarray(pairwise_sq_dists(jnp.asarray(A_b[:20]), jnp.asarray(A_b[20:60])))
D_ref = ((A_b[:20, None, :] - A_b[None, 20:60, :]) ** 2).sum(-1)
compare("pairwise squared distances", D_ours, D_ref)
rows = []
for name, (A, B, ya, yb_) in {"breast cancer": (A_b, B_b, yb[trb], yb[teb]), "german credit": (A_g, B_g, yg[trg], yg[teg])}.items():
    for k in (1, 5, 15):
        for w in ("uniform", "distance"):
            ours = knn_predict(A, ya, B, k, 2, w)
            ref = KNeighborsClassifier(k, weights=w).fit(A, ya)
            rows.append((name, k, w, float(np.mean(ours == yb_)), float(ref.score(B, yb_)), float(np.mean(ours == ref.predict(B)))))
print(pl.DataFrame(rows, schema=["data", "k", "weights", "acc ours", "acc sklearn", "same pred."], orient="row"))

**Scaling, $k$ and irrelevant features** (all with your `knn_predict`): the accuracy with raw and standardized features,
the curve of $k$ on the moons, and the accuracy when $r$ columns of noise are added to the breast-cancer data.

In [ ]:
raw = np.mean(knn_predict(Xb[trb], yb[trb], Xb[teb], 5, 2) == yb[teb])
std = np.mean(knn_predict(A_b, yb[trb], B_b, 5, 2) == yb[teb])
print(f"breast cancer, k = 5: raw features {raw:.3f}, standardized {std:.3f}")

Xm, ym = make_moons(300, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(3000, noise=0.3, random_state=1)
ks = [1, 3, 5, 9, 15, 25, 51, 101, 201]
tr_acc = [np.mean(knn_predict(Xm, ym, Xm, k, 2) == ym) for k in ks]
te_acc = [np.mean(knn_predict(Xm, ym, Xm_te, k, 2) == ym_te) for k in ks]
plot_curves(ks, {"train": np.array(tr_acc), "test": np.array(te_acc)}, "k", logx=True)

rng = np.random.default_rng(0)
rows = []
for r in (0, 10, 50, 200, 1000):
    A = np.c_[A_b, rng.normal(size=(len(A_b), r))]
    B = np.c_[B_b, rng.normal(size=(len(B_b), r))]
    knn_acc = np.mean(knn_predict(A, yb[trb], B, 5, 2) == yb[teb])
    tree_acc = np.mean(predict_tree(fit_tree(A, yb[trb], 2, max_depth=4, min_leaf=5), B) == yb[teb])
    lr = LogisticRegression(max_iter=3000).fit(A, yb[trb]).score(B, yb[teb])
    rows.append((r, float(knn_acc), float(tree_acc), float(lr)))
print(pl.DataFrame(rows, schema=["noise features", "KNN k=5", "our tree", "logistic"], orient="row"))

# Part G — KNN for regression

## G1. The local mean

Implement `knn_regress(Xtr, ytr, Xte, k)`: the mean target of the $k$ nearest training points (reuse
`pairwise_sq_dists` and `jax.lax.top_k`).

In [ ]:
def knn_regress(Xtr, ytr, Xte, k=5):
    raise NotImplementedError

In [ ]:
mu_c, sd_c = fit_standardizer(Xtr_c)
A_c, B_c = standardize(Xtr_c, mu_c, sd_c), standardize(Xte_c, mu_c, sd_c)
rows = []
for k in (1, 3, 5, 10, 20, 50):
    p = knn_regress(A_c, ytr_c, B_c, k)
    ref = KNeighborsRegressor(k).fit(A_c, ytr_c).predict(B_c)
    rows.append((k, float(np.sqrt(np.mean((p - yte_c) ** 2))), float(np.sqrt(np.mean((ref - yte_c) ** 2))), float(np.max(np.abs(p - ref)))))
print(pl.DataFrame(rows, schema=["k", "RMSE ours", "RMSE sklearn", "max |diff|"], orient="row"))
print(f"concrete, test RMSE: linear {np.sqrt(np.mean((lin.predict(Xte_c) - yte_c) ** 2)):.2f}, "
      f"tree (depth 10) {np.sqrt(np.mean((predict_reg_tree(fit_reg_tree(Xtr_c, ytr_c, 10, 5), Xte_c) - yte_c) ** 2)):.2f}, "
      f"KNN k=5 {np.sqrt(np.mean((knn_regress(A_c, ytr_c, B_c, 5) - yte_c) ** 2)):.2f} MPa")

# Part H — Challenge: the symbolist and the analogizer (home)

Choose a data set of Project 1. Compare a **decision tree** (with `min_samples_leaf` and `ccp_alpha` chosen by
cross-validation) and a **KNN** (the number of neighbours chosen by cross-validation, standardization inside the pipeline)
with logistic regression on repeated stratified splits, and decide with the corrected paired $t$-test of Class 05
whether any difference is real.

## H1. Repeated splits

Implement `repeated_scores(make_model, X, y, n_rep, test_frac)`: for `rep in range(n_rep)` split with
`stratified_split(y, test_frac, np.random.default_rng(rep))`, standardize with the **training** statistics, fit
`make_model()` and return the array of test accuracies.

In [ ]:
def repeated_scores(make_model, X, y, n_rep=10, test_frac=0.25):
    raise NotImplementedError

In [ ]:
results = {
    "logistic": repeated_scores(lambda: LogisticRegression(max_iter=3000), Xg, yg),
    "tree (tuned)": repeated_scores(lambda: GridSearchCV(DecisionTreeClassifier(random_state=0), {"min_samples_leaf": [1, 5, 10, 20, 50]}, cv=5), Xg, yg),
    "KNN (tuned)": repeated_scores(lambda: GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [1, 3, 5, 9, 15, 25]}, cv=5), Xg, yg),
}
print(pl.DataFrame([(k, float(v.mean()), float(v.std(ddof=1))) for k, v in results.items()], schema=["German credit", "mean accuracy", "sd"], orient="row"))
for a, b in (("KNN (tuned)", "tree (tuned)"), ("KNN (tuned)", "logistic"), ("tree (tuned)", "logistic")):
    d = results[a] - results[b]
    t = d.mean() / np.sqrt((1 / len(d) + 1 / 3) * d.var(ddof=1))
    print(f"{a} - {b}: mean difference {d.mean():+.4f}, t = {t:.2f}, p = {2 * stats.t.sf(abs(t), len(d) - 1):.3f}")